# Testing pulling data for active/nonactive gages

In [61]:
import pandas as pd
import geopandas as gpd
import requests
import subprocess
from pathlib import Path
import shutil
import time
from io import StringIO
import folium
from IPython.display import display

#### Inspecting nwis 30_data_summarizes github

Need to find where they are pulling sources for this .R code https://github.com/DOI-USGS/national-flow-observations/blob/main/30_data_summarize/src/summarize_active_flow_gages.R 

gage activity history (Pulls gage ID +years active (using usgs classification by days active))

gage location (ID+lat,lon,state)

In [43]:
url = "https://api.github.com/repos/DOI-USGS/national-flow-observations/contents/30_data_summarize.yml"

response = requests.get(url, timeout=30)
print("Status:", response.status_code)

if response.status_code == 200:
    print(response.json().get("download_url"))
    print(requests.get(response.json()["download_url"], timeout=30).text)
else:
    print(response.text[:1000])

Status: 200
https://raw.githubusercontent.com/DOI-USGS/national-flow-observations/main/30_data_summarize.yml
target_default: 30_data_summarize

packages:
  - scipiper
  - dplyr
  - purrr
  - dataRetrieval

sources:
  - 30_data_summarize/src/summarize_active_flow_gages.R
  - 30_data_summarize/src/record_diagnostics_active_flow_gages.R
  - 30_data_summarize/src/site_summary.R

targets:
  30_data_summarize:
    depends:
      - 30_data_summarize/out/active_flow_gages.rds.ind
      - 30_data_summarize/out/active_flow_gages_summary.rds.ind
      - 30_data_summarize/out/active_flow_gages_diagnostics.png
      - 30_data_summarize/out/active_flow_gages_wy.rds.ind
      - 30_data_summarize/out/active_flow_gages_summary_wy.rds.ind
      - 30_data_summarize/out/active_flow_gages_diagnostics_wy.png

  # tally how many active flow gages there were per year
  30_data_summarize/out/active_flow_gages.rds.ind:
    command: tally_active_flow_gages(
      out_ind = target_name,
      all_dat_ind = '20_da

#### KEY FILES IDENTIFIED AS:
##### active_flow_gages_wy.rds = Active gage IDs grouped by water year
##### active_flow_gages_summary_wy.rds = Summary of each gage's active years and gaps
##### observation_summary.rds = A separate observation summary that may help investigate data coverage

##### and min days active defined as 335 days/year

##### NEXT DOWNLOADING THESE THREE FILES TO data/raw FOLDER

##### -only have summary_wy.rds file from gages github
##### the other two are stored as .rd.ind (indicator files in nwis github. (need to find original source))

In [44]:
project_folder = Path.cwd().parent
raw_folder = project_folder / "data" / "raw"
raw_folder.mkdir(parents=True, exist_ok=True)

files = {
    "active_flow_gages_summary_wy.rds":
        "https://raw.githubusercontent.com/DOI-USGS/gages-through-the-ages/main/gage_timeseries_gif/active_flow_gages_summary_wy.rds",
    "active_flow_gages_wy.rds":
        "https://raw.githubusercontent.com/DOI-USGS/gages-through-the-ages/main/gage_timeseries_gif/active_flow_gages_wy.rds",
}

for filename, url in files.items():
    response = requests.get(url, timeout=60)
    print(filename, "| HTTP", response.status_code)

    if response.status_code == 200:
        path = raw_folder / filename
        path.write_bytes(response.content)
        print("  Saved:", path)
        print("  Size:", path.stat().st_size, "bytes")
    else:
        print("  Not available at this URL; not saved.")

active_flow_gages_summary_wy.rds | HTTP 200
  Saved: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/raw/active_flow_gages_summary_wy.rds
  Size: 269955 bytes
active_flow_gages_wy.rds | HTTP 404
  Not available at this URL; not saved.


#### active_flow_gages_wy.rds not available
##### Going to inspect the summary file and then search for this file if its needed

##### Reading active_flow_gages_summary_wy.rds using R and export as csv for to python to inspect (stored in data/intermediate)

In [45]:
project_folder = Path.cwd().parent
rds_path = project_folder / "data" / "raw" / "active_flow_gages_summary_wy.rds"
csv_path = project_folder / "data" / "intermediate" / "active_flow_gages_summary_wy.csv"
script_path = project_folder / "data" / "intermediate" / "read_gage_summary.R"

r_script = f'''
data <- readRDS("{rds_path.as_posix()}")

collapse_list <- function(x) {{
  if (is.null(x) || length(x) == 0 || all(is.na(x))) {{
    return(NA_character_)
  }}
  paste(x[!is.na(x)], collapse = ";")
}}

data$which_years_active <- vapply(
  data$which_years_active, collapse_list, character(1)
)

data$gap_years <- vapply(
  data$gap_years, collapse_list, character(1)
)

write.csv(data, "{csv_path.as_posix()}", row.names = FALSE)
cat("Exported rows:", nrow(data), "\\\\n")
cat("Exported columns:", ncol(data), "\\\\n")
'''

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    ["Rscript", str(script_path)],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
print("CSV exists:", csv_path.exists())
if csv_path.exists():
    print("CSV size:", csv_path.stat().st_size, "bytes")

Return code: 0
Exported rows: 23242 \nExported columns: 6 \n

CSV exists: True
CSV size: 3910060 bytes


#### Inspecting the summary csv

In [46]:
project_folder = Path.cwd().parent
csv_path = project_folder / "data" / "intermediate" / "active_flow_gages_summary_wy.csv"

gages = pd.read_csv(csv_path)

print("Rows and columns:", gages.shape)
print("\nColumn names:")
print(gages.columns.tolist())

print("\nFirst 5 rows:")
display(gages.head())

print("\nData types:")
print(gages.dtypes)

print("\nExample activity histories:")
for _, row in gages.head(5).iterrows():
    print(
        "Site:", row["site"],
        "| Earliest active year:", row["earliest_active_year"],
        "| Years active:", str(row["which_years_active"])[:250]
    )

Rows and columns: (23242, 6)

Column names:
['site', 'n_years_active', 'earliest_active_year', 'which_years_active', 'any_gaps', 'gap_years']

First 5 rows:


,site,n_years_active,earliest_active_year,which_years_active,any_gaps,gap_years
0,7010000,159,1863,1863;1864;1865;1866;1867;1868;1869;1870;1871;1...,False,NaN
1,5420500,148,1874,1874;1875;1876;1877;1878;1879;1880;1881;1882;1...,False,NaN
2,3568000,134,1875,1875;1876;1877;1878;1879;1880;1881;1882;1883;1...,False,NaN
3,3193000,144,1878,1878;1879;1880;1881;1882;1883;1884;1885;1886;1...,False,NaN
4,5474500,143,1879,1879;1880;1881;1882;1883;1884;1885;1886;1887;1...,False,NaN



Data types:
site                    int64
n_years_active          int64
earliest_active_year    int64
which_years_active        str
any_gaps                 bool
gap_years                 str
dtype: object

Example activity histories:
Site: 7010000 | Earliest active year: 1863 | Years active: 1863;1864;1865;1866;1867;1868;1869;1870;1871;1872;1873;1874;1875;1876;1877;1878;1879;1880;1881;1882;1883;1884;1885;1886;1887;1888;1889;1890;1891;1892;1893;1894;1895;1896;1897;1898;1899;1900;1901;1902;1903;1904;1905;1906;1907;1908;1909;1910;1911;1912;
Site: 5420500 | Earliest active year: 1874 | Years active: 1874;1875;1876;1877;1878;1879;1880;1881;1882;1883;1884;1885;1886;1887;1888;1889;1890;1891;1892;1893;1894;1895;1896;1897;1898;1899;1900;1901;1902;1903;1904;1905;1906;1907;1908;1909;1910;1911;1912;1913;1914;1915;1916;1917;1918;1919;1920;1921;1922;1923;
Site: 3568000 | Earliest active year: 1875 | Years active: 1875;1876;1877;1878;1879;1880;1881;1882;1883;1884;1885;1886;1887;1888;1889;1890;1891;

#### NOW TESTING USGS WATER DATA API LOCATIONS SOURCE https://api.waterdata.usgs.gov/ogcapi/v1/openapi#/monitoring-locations/getMonitoring-locationsFeatures

In [54]:
url = (
    "https://api.waterdata.usgs.gov/ogcapi/v1/"
    "collections/monitoring-locations/items"
)


#### seeing if water data usgs api has better location matches than GAGES 3

In [48]:
project_dir = Path.cwd().parent
activity_file = project_dir / "data" / "intermediate" / "active_flow_gages_summary_wy.csv"

# Load the activity summary preserving station IDs as text
activity = pd.read_csv(activity_file, dtype={"site": str})
activity["site"] = activity["site"].str.strip()

# Standardize IDs for the USGS API
site_ids = activity["site"].dropna().unique().tolist()
site_ids = [s.removeprefix("USGS-").strip() for s in site_ids if s.strip()]

print(f"Activity-summary stations to look up: {len(site_ids):,}")

# USGS Water Data API monitoring locations
url = "https://api.waterdata.usgs.gov/ogcapi/v0/collections/monitoring-locations/items"

found_records = []
failed_batches = []
batch_size = 100

for start in range(0, len(site_ids), batch_size):
    batch = site_ids[start:start + batch_size]

    params = {
        "monitoring_location_id": ",".join(
            s if s.startswith("USGS-") else f"USGS-{s}" for s in batch
        ),
        "limit": batch_size
    }

    try:
        response = requests.get(url, params=params, timeout=60)

        if response.status_code == 200:
            data = response.json()
            features = data.get("features", [])
            found_records.extend(features)
        else:
            failed_batches.append({
                "batch_start": start,
                "status_code": response.status_code,
                "message": response.text[:200]
            })

    except requests.RequestException as e:
        failed_batches.append({
            "batch_start": start,
            "status_code": "Request error",
            "message": str(e)[:200]
        })

    if (start // batch_size + 1) % 20 == 0:
        print(f"Processed {min(start + batch_size, len(site_ids)):,} of {len(site_ids):,}")

    time.sleep(0.1)

print(f"Unique activity-summary IDs: {len(site_ids):,}")
print(f"Location records returned: {len(found_records):,}")
print(f"Failed batches: {len(failed_batches):,}")

if found_records:
    locations_preview = pd.json_normalize(found_records)
    print("\nAvailable location fields:")
    print(locations_preview.columns.tolist())
    display(locations_preview.head())

if failed_batches:
    print("\nFirst failed batches:")
    display(pd.DataFrame(failed_batches).head(10))

Activity-summary stations to look up: 23,242
Processed 2,000 of 23,242
Processed 4,000 of 23,242
Processed 6,000 of 23,242
Processed 8,000 of 23,242
Processed 10,000 of 23,242
Processed 12,000 of 23,242
Processed 14,000 of 23,242
Processed 16,000 of 23,242
Processed 18,000 of 23,242
Processed 20,000 of 23,242
Processed 22,000 of 23,242
Unique activity-summary IDs: 23,242
Location records returned: 0
Failed batches: 233

First failed batches:


,batch_start,status_code,message
0,0,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
1,100,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
2,200,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
3,300,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
4,400,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
5,500,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
6,600,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
7,700,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
8,800,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."
9,900,400,"{\n ""code"": ""InvalidQuery"",\n ""type"": ""Inval..."


#### GET API KEY FOR USGS

In [49]:
api_key = "mPgnnQwJbw7q6PW6rqQpkpBCGzMLwqdjcCLBrnw8"

test_ids = activity["site"].dropna().astype(str).str.strip().head(10).tolist()

params = {
    "id": ",".join(
        s if s.startswith("USGS-") else f"USGS-{s}"
        for s in test_ids
    ),
    "limit": 100
}

url = "https://api.waterdata.usgs.gov/ogcapi/v0/collections/monitoring-locations/items"

response = requests.get(
    url,
    params=params,
    headers={"X-Api-Key": api_key},
    timeout=60
)

print("Status:", response.status_code)
print("Remaining requests:", response.headers.get("X-RateLimit-Remaining"))
print("Response preview:", response.text[:1000])

if response.status_code == 200:
    test_data = response.json()
    print("Locations returned:", len(test_data.get("features", [])))
    if test_data.get("features"):
        display(pd.json_normalize(test_data["features"]).head())

Status: 200
Remaining requests: 748
Response preview: {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "id": "USGS-01460000",
        "agency_code": "USGS",
        "agency_name": "U.S. Geological Survey",
        "monitoring_location_number": "01460000",
        "monitoring_location_name": "Tohickon Creek at Point Pleasant, PA",
        "district_code": "42",
        "country_code": "US",
        "country_name": "United States of America",
        "state_code": "42",
        "state_name": "Pennsylvania",
        "county_code": "017",
        "county_name": "Bucks County",
        "minor_civil_division_code": null,
        "site_type_code": "ST",
        "site_type": "Stream",
        "hydrologic_unit_code": "020401050908",
        "basin_code": null,
        "altitude": 70.0,
        "altitude_accuracy": 5.0,
        "altitude_method_code": "M",
        "altitude_method_name": "Interpolated from topographic map.",
        "v

,type,id,properties.id,properties.agency_code,properties.agency_name,properties.monitoring_location_number,properties.monitoring_location_name,properties.district_code,properties.country_code,properties.country_name,...,properties.national_aquifer_code,properties.aquifer_type_code,properties.well_constructed_depth,properties.hole_constructed_depth,properties.depth_source_code,properties.revision_note,properties.revision_created,properties.revision_modified,geometry.type,geometry.coordinates
0,Feature,USGS-01460000,USGS-01460000,USGS,U.S. Geological Survey,01460000,"Tohickon Creek at Point Pleasant, PA",42,US,United States of America,...,None,None,None,None,None,NaN,NaN,NaN,Point,"[-75.0662818138315, 40.4237156143963]"
1,Feature,USGS-02197000,USGS-02197000,USGS,U.S. Geological Survey,02197000,"SAVANNAH RIVER AT AUGUSTA, GA",45,US,United States of America,...,None,None,None,None,None,&lt;p&gt;WSP 1303: 1927-39 (monthly runoff). W...,2017-11-13T06:00:00+00:00,2021-07-15T16:15:05.187000+00:00,Point,"[-81.9420833333333, 33.3725277777778]"
2,Feature,USGS-03193000,USGS-03193000,USGS,U.S. Geological Survey,03193000,"KANAWHA RIVER AT KANAWHA FALLS, WV",54,US,United States of America,...,None,None,None,None,None,"WSP 923: 1878, 1886, 1897, 1899, 1901-03. WSP ...",2017-11-13T06:00:00+00:00,NaN,Point,"[-81.214274466564, 38.1381632299295]"
3,Feature,USGS-03568000,USGS-03568000,USGS,U.S. Geological Survey,03568000,"TENNESSEE RIVER AT CHATTANOOGA, TN",47,US,United States of America,...,None,None,None,None,None,WSP 353: 1874-1912. WSP 783: 1917. WSP 823: 18...,2017-11-13T06:00:00+00:00,NaN,Point,"[-85.2784298642281, 35.0867755467101]"
4,Feature,USGS-05082500,USGS-05082500,USGS,U.S. Geological Survey,05082500,"RED RIVER OF THE NORTH AT GRAND FORKS, ND",38,US,United States of America,...,None,None,None,None,None,WSP 855: 1936(M). WSP 1115: 1942. WSP 1175: 18...,2017-11-13T06:00:00+00:00,NaN,Point,"[-97.028889, 47.926944]"


#### Matching locations to usgs water data in batches CREATING LOCATIONS FILE

In [50]:
project_dir = Path.cwd().parent
output_dir = project_dir / "data" / "intermediate"
output_dir.mkdir(parents=True, exist_ok=True)

locations_file = output_dir / "usgs_api_locations.csv"
failed_file = output_dir / "usgs_api_failed_batches.csv"
missing_file = output_dir / "activity_summary_ids_missing_api.csv"

# Get all unique station IDs from the activity summary
site_ids = (
    activity["site"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

site_ids = [
    s.removeprefix("USGS-")
    for s in site_ids
    if s and s.lower() != "nan"
]

url = "https://api.waterdata.usgs.gov/ogcapi/v0/collections/monitoring-locations/items"
headers = {"X-Api-Key": api_key}
batch_size = 100

location_records = []
failed_batches = []

for start in range(0, len(site_ids), batch_size):
    batch = site_ids[start:start + batch_size]

    params = {
        "id": ",".join(f"USGS-{s}" for s in batch),
        "limit": batch_size
    }

    success = False

    
    for attempt in range(3):
        try:
            response = requests.get(
                url, params=params, headers=headers, timeout=60
            )

            if response.status_code == 200:
                features = response.json().get("features", [])

                for feature in features:
                    properties = feature.get("properties", {})
                    coordinates = (feature.get("geometry") or {}).get(
                        "coordinates", []
                    )

                    location_records.append({
                        "site": str(
                            properties.get("monitoring_location_number")
                            or properties.get("id", "")
                        ).removeprefix("USGS-"),
                        "api_id": properties.get("id"),
                        "station_name": properties.get(
                            "monitoring_location_name"
                        ),
                        "state_name": properties.get("state_name"),
                        "latitude": (
                            coordinates[1] if len(coordinates) >= 2 else None
                        ),
                        "longitude": (
                            coordinates[0] if len(coordinates) >= 2 else None
                        )
                    })

                success = True
                break

            if response.status_code == 429 or response.status_code >= 500:
                time.sleep(2 ** (attempt + 1))
            else:
                break

        except requests.RequestException:
            time.sleep(2 ** (attempt + 1))

    if not success:
        failed_batches.append({
            "batch_start": start,
            "batch_size": len(batch),
            "status_code": (
                response.status_code if "response" in locals() else None
            ),
            "site_ids": ";".join(batch)
        })

    # checkpoints every 10 batches
    if (start // batch_size + 1) % 10 == 0:
        pd.DataFrame(location_records).drop_duplicates(
            subset="site"
        ).to_csv(locations_file, index=False)

        pd.DataFrame(failed_batches).to_csv(failed_file, index=False)

        print(
            f"Processed {min(start + batch_size, len(site_ids)):,}"
            f" / {len(site_ids):,} station IDs; "
            f"locations returned: {len(location_records):,}; "
            f"failed batches: {len(failed_batches)}"
        )

    time.sleep(0.2)


locations = pd.DataFrame(location_records)

if not locations.empty:
    locations = locations.drop_duplicates(subset="site")
    locations.to_csv(locations_file, index=False)

pd.DataFrame(failed_batches).to_csv(failed_file, index=False)


found_ids = set(locations["site"]) if not locations.empty else set()
missing_ids = sorted(set(site_ids) - found_ids)

pd.DataFrame({"site": missing_ids}).to_csv(missing_file, index=False)


print(f"Activity-summary stations: {len(site_ids):,}")
print(f"Unique locations returned: {len(found_ids):,}")
print(f"Stations without returned locations: {len(missing_ids):,}")
print(f"Failed batches: {len(failed_batches):,}")
print(f"\nLocations saved to: {locations_file}")
print(f"Missing IDs saved to: {missing_file}")
print(f"Failed batches saved to: {failed_file}")

Processed 1,000 / 23,242 station IDs; locations returned: 1,000; failed batches: 0
Processed 2,000 / 23,242 station IDs; locations returned: 2,000; failed batches: 0
Processed 3,000 / 23,242 station IDs; locations returned: 3,000; failed batches: 0
Processed 4,000 / 23,242 station IDs; locations returned: 4,000; failed batches: 0
Processed 5,000 / 23,242 station IDs; locations returned: 5,000; failed batches: 0
Processed 6,000 / 23,242 station IDs; locations returned: 6,000; failed batches: 0
Processed 7,000 / 23,242 station IDs; locations returned: 7,000; failed batches: 0
Processed 8,000 / 23,242 station IDs; locations returned: 8,000; failed batches: 0
Processed 9,000 / 23,242 station IDs; locations returned: 9,000; failed batches: 0
Processed 10,000 / 23,242 station IDs; locations returned: 10,000; failed batches: 0
Processed 11,000 / 23,242 station IDs; locations returned: 11,000; failed batches: 0
Processed 12,000 / 23,242 station IDs; locations returned: 12,000; failed batches: 

#### Join activity summary to location data (made in previous step)

In [51]:
project_dir = Path.cwd().parent
data_dir = project_dir / "data" / "intermediate"

activity_file = data_dir / "active_flow_gages_summary_wy.csv"
locations_file = data_dir / "usgs_api_locations.csv"


activity = pd.read_csv(activity_file, dtype={"site": str})
locations = pd.read_csv(locations_file, dtype={"site": str})


activity["site"] = activity["site"].str.strip().str.removeprefix("USGS-")
locations["site"] = locations["site"].str.strip().str.removeprefix("USGS-")

# Check for duplicate IDs before joining
print("Activity rows:", len(activity))
print("Unique activity IDs:", activity["site"].nunique())
print("Location rows:", len(locations))
print("Unique location IDs:", locations["site"].nunique())

# Keep every activity-summary station
master = activity.merge(
    locations,
    on="site",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Identify stations with missing locations or coordinates
no_location = master["_merge"] == "left_only"
no_coordinates = master["latitude"].isna() | master["longitude"].isna()

master["location_found"] = ~no_location
master["coordinates_available"] = ~no_coordinates

# Save the joined master table
master_file = data_dir / "gages_activity_with_api_locations.csv"
master.drop(columns="_merge").to_csv(master_file, index=False)



print("Master table rows:", len(master))
print("Stations matched to API:", (~no_location).sum())
print("Stations without API match:", no_location.sum())
print("Stations missing coordinates:", no_coordinates.sum())
print("Output saved to:", master_file)


display(master.loc[no_location, ["site", "n_years_active", "which_years_active"]])


display(master[[
    "site", "station_name", "state_name",
    "latitude", "longitude", "n_years_active",
    "earliest_active_year", "which_years_active",
    "location_found", "coordinates_available"
]].head())

Activity rows: 23242
Unique activity IDs: 23242
Location rows: 23241
Unique location IDs: 23241
Master table rows: 23242
Stations matched to API: 23241
Stations without API match: 1
Stations missing coordinates: 4
Output saved to: /Users/sophiameyers/Documents/GitHub/USGS-Gages-Research/data/intermediate/gages_activity_with_api_locations.csv


,site,n_years_active,which_years_active
16580,09489082,6,1986;1987;1988;1989;1990;1991


,site,station_name,state_name,latitude,longitude,n_years_active,earliest_active_year,which_years_active,location_found,coordinates_available
0,07010000,"Mississippi River at St. Louis, MO",Missouri,38.629000,-90.179778,159,1863,1863;1864;1865;1866;1867;1868;1869;1870;1871;1...,True,True
1,05420500,"Mississippi River at Clinton, IA",Iowa,41.780586,-90.252073,148,1874,1874;1875;1876;1877;1878;1879;1880;1881;1882;1...,True,True
2,03568000,"TENNESSEE RIVER AT CHATTANOOGA, TN",Tennessee,35.086776,-85.278430,134,1875,1875;1876;1877;1878;1879;1880;1881;1882;1883;1...,True,True
3,03193000,"KANAWHA RIVER AT KANAWHA FALLS, WV",West Virginia,38.138163,-81.214274,144,1878,1878;1879;1880;1881;1882;1883;1884;1885;1886;1...,True,True
4,05474500,"Mississippi River at Keokuk, IA",Iowa,40.393655,-91.374318,143,1879,1879;1880;1881;1882;1883;1884;1885;1886;1887;1...,True,True


#### Missing coordinates for 4 stations and missing api match for 1 station
###### Inspecting these gage sites

In [52]:
missing_sites = ["11516100", "09489082", "421247071470201", "422108071052501"]

url = "https://waterservices.usgs.gov/nwis/site/"

params = {
    "format": "rdb",
    "sites": ",".join(missing_sites),
    "siteOutput": "expanded"
}

response = requests.get(url, params=params, timeout=60)

print("Status:", response.status_code)
print("URL:", response.url)

if response.status_code == 200:
    lines = [
        line for line in response.text.splitlines()
        if not line.startswith("#")
        and line.strip()
    ]

    print("\nFirst lines returned:")
    print("\n".join(lines[:8]))

    # RDB files have two header lines before the data
    from io import StringIO

    rdb_lines = [
        line for line in response.text.splitlines()
        if not line.startswith("#") and line.strip()
    ]

    if len(rdb_lines) >= 3:
        site_data = pd.read_csv(
            StringIO("\n".join(rdb_lines)),
            sep="\t",
            dtype=str,
            skiprows=[1]
        )

        print("\nColumns containing coordinates:")
        print([
            col for col in site_data.columns
            if "lat" in col.lower() or "long" in col.lower()
        ])

        display(site_data[[
            col for col in [
                "site_no",
                "station_nm",
                "dec_lat_va",
                "dec_long_va"
            ]
            if col in site_data.columns
        ]])
else:
    print(response.text[:1000])

Status: 200
URL: https://waterservices.usgs.gov/nwis/site/?format=rdb&sites=11516100%2C09489082%2C421247071470201%2C422108071052501&siteOutput=expanded

First lines returned:
agency_cd	site_no	station_nm	site_tp_cd	lat_va	long_va	dec_lat_va	dec_long_va	coord_meth_cd	coord_acy_cd	coord_datum_cd	dec_coord_datum_cd	district_cd	state_cd	county_cd	country_cd	land_net_ds	map_nm	map_scale_fc	alt_va	alt_meth_cd	alt_acy_va	alt_datum_cd	huc_cd	basin_cd	topo_cd	instruments_cd	construction_dt	inventory_dt	drain_area_va	contrib_drain_area_va	tz_cd	local_time_fg	reliability_cd	gw_file_cd	nat_aqfr_cd	aqfr_cd	aqfr_type_cd	well_depth_va	hole_depth_va	depth_src_cd	project_no
5s	15s	50s	7s	16s	16s	16s	16s	1s	1s	10s	10s	3s	2s	3s	2s	23s	20s	7s	8s	1s	3s	10s	16s	2s	1s	30s	8s	8s	8s	8s	6s	1s	1s	30s	10s	8s	1s	8s	8s	1s	12s
USFS	09489082	NORTH FORK THOMAS CREEK NEAR ALPINE, AZ	ST	334030.1	1091612.4	33.67502778	-109.2701111	N	S	NAD83	NAD83	04	04	011	US				 8380.	M	 20	NGVD29	15060101			YNNNYNNNNNYNNYNNNNNNNNNNNNNN

,site_no,station_nm,dec_lat_va,dec_long_va
0,09489082,"NORTH FORK THOMAS CREEK NEAR ALPINE, AZ",33.67502778,-109.2701111
1,11516100,"GREEN SPRINGS POWERPLANT DIVER NR ASHLAND,OREG.",NaN,NaN
2,421247071470201,"BRIDGE-RT20W-QUINSIGAMOND, MA",NaN,NaN
3,422108071052501,"BRIDGE-RT2AE-BOSTON, MA",NaN,NaN


#### ^ only found 09489082 site coordinates, need to find another source for the other three missing gages (https://waterservices.usgs.gov/nwis/site/?format=rdb&sites=11516100%2C09489082%2C421247071470201%2C422108071052501&siteOutput=expanded)

###### moving on without these three gages

### Joining locations with activity summary of gages (reorganizing table to give states and years as rows with active or not active per row) Need to organize like master file, state by state

### CHECK THE OUTPUT TABLES 
#### see if the mapping table is necessary and maybe delete (descirbe specific outputs differences)

## Mapping master dataset with state and year filters interchangable

#### Had to fix my "Master" file to remove not active yet so the file is smalled (moving master file to my local computer desktop in folder called EMLAB BOX fies)